# ERP AI Assistant

An AI-powered assistant that enables users to interact with ERP database systems using natural language instead of writing SQL queries manually.

The project is built as an independent backend API using **FastAPI, Python, Ollama, MySQL, SQLAlchemy, and SQLGlot**. It provides a foundation for intelligent database querying, SQL validation, query execution, natural-language responses, and context-aware conversations.

---

## 1. Project Overview

Traditional ERP systems require users to navigate multiple screens, understand database-related information, or depend on technical team members to retrieve specific data.

The ERP AI Assistant aims to simplify this interaction.

Users can ask questions in natural language, such as:

- Show me the top 5 customers.
- List all products in a specific category.
- Show the emails of those customers.
- Which suppliers are available?
- Retrieve orders with their related customer information.

The assistant processes the question, generates an SQL query, validates it, executes it against the connected MySQL database, and converts the returned records into a clear natural-language answer.

The system also supports conversation history, allowing users to ask follow-up questions while preserving the context of previous interactions.

### Main Objectives

- Natural-language interaction with ERP databases.
- Automated MySQL query generation.
- SQL validation and security controls.
- Controlled database query execution.
- Natural-language result generation.
- Context-aware conversations.
- Future integration with existing Laravel ERP systems.
- Future document-based knowledge retrieval using RAG.

---

## 2. Technology Stack

| Component | Technology |
|---|---|
| Backend Framework | FastAPI |
| Programming Language | Python 3.13 |
| Local LLM Runtime | Ollama |
| Language Model | Qwen3 4B |
| Database | MySQL / MariaDB |
| Database ORM | SQLAlchemy |
| Database Driver | PyMySQL |
| SQL Parsing & Validation | SQLGlot |
| Configuration | Pydantic Settings |
| API Documentation | Swagger / OpenAPI |
| API Testing | Swagger UI |
| Environment | Python Virtual Environment |

### Selected Language Model

The current development model is:

`qwen3:4b-instruct-2507-q4_K_M`

It runs locally through Ollama.

This model was selected as an initial option based on the available development hardware:

- 16 GB RAM.
- CPU-only environment.
- Lenovo ThinkPad AMD laptop.

The model selection can be changed later depending on response quality, speed, memory usage, and deployment hardware.

---

## 3. System Architecture

The application is designed as an independent service, separate from the Laravel ERP backend.

### High-Level Architecture

```text
                  User
                   |
                   v
             Natural Language
                Question
                   |
                   v
              FastAPI API
                   |
                   v
          Assistant Service Layer
                   |
                   v
          Conversation Context
             (When Needed)
                   |
                   v
             Ollama / Qwen3
                   |
                   v
             Text-to-SQL
                   |
                   v
          SQL Validation Layer
                SQLGlot
                   |
                   v
             MySQL Database
                   |
                   v
             Query Results
                   |
                   v
          Answer Generation
                   |
                   v
          Natural Language Answer
                   |
                   v
          Conversation History
                   |
                   v
                 User
```

### Main Processing Workflow

1. Receive the user's natural-language question.
2. Retrieve the database schema.
3. Retrieve previous conversation messages when using the Conversation Assistant.
4. Build a schema-aware prompt.
5. Send the prompt to the local LLM through Ollama.
6. Generate a MySQL SELECT query.
7. Validate the generated SQL using SQLGlot.
8. Execute the validated query through SQLAlchemy.
9. Retrieve the database records.
10. Send the query results to the LLM.
11. Generate a natural-language answer.
12. Save conversation messages when conversation mode is enabled.
13. Return the final response through FastAPI.

---

## 4. Project Structure

The current project follows a modular service-based architecture.

```text
ERP_ASS/
│
├── app/
│   │
│   ├── main.py
│   │
│   ├── api/
│   │   └── v1/
│   │       ├── router.py
│   │       └── endpoints/
│   │           ├── health.py
│   │           ├── models.py
│   │           ├── schema.py
│   │           ├── sql.py
│   │           ├── assistant.py
│   │           ├── conversations.py
│   │           └── conversation_assistant.py
│   │
│   ├── core/
│   │   ├── config.py
│   │   └── database.py
│   │
│   ├── schemas/
│   │   ├── ollama.py
│   │   ├── sql.py
│   │   └── conversation_assistant.py
│   │
│   ├── models/
│   │   ├── base.py
│   │   ├── conversation.py
│   │   └── conversation_message.py
│   │
│   └── services/
│       ├── ollama_service.py
│       ├── schema_context_service.py
│       ├── sql_generation_service.py
│       ├── sql_validation_service.py
│       ├── sql_execution_service.py
│       ├── answer_generation_service.py
│       ├── sql_assistant_service.py
│       ├── conversation_service.py
│       ├── conversation_prompt_service.py
│       ├── conversation_sql_generation_service.py
│       └── conversation_assistant_service.py
│
├── .venv/
├── .env
├── .gitignore
├── requirements.txt
├── README.md
└── LICENSE
```

*Note: This structure represents the implemented project organization. Additional modules and folders may be introduced as development continues.*

### Architecture Responsibilities

**API Layer**

Responsible for receiving HTTP requests, validating request data, calling the appropriate services, and returning API responses.

**Core Layer**

Contains application configuration and database connection management.

**Schemas Layer**

Defines request and response structures using Pydantic.

**Models Layer**

Defines SQLAlchemy database models for application-managed data.

**Services Layer**

Contains the business logic, including LLM communication, SQL generation, validation, execution, answer generation, and conversation management.

---

## 5. Database Design

The project currently uses a dedicated development database:

`erp_ai_assistant_test`

The test database contains 10 tables and 162 records.

### ERP Test Database Tables

| Table | Records | Description |
|---|---:|---|
| customers | 10 | Customer information |
| categories | 5 | Product categories |
| suppliers | 8 | Supplier information |
| products | 20 | Product records |
| orders | 20 | Customer orders |
| order_items | 40 | Order line items |
| invoices | 20 | Invoice records |
| payments | 15 | Payment records |
| purchase_orders | 8 | Purchase orders |
| purchase_order_items | 16 | Purchase order details |
| **Total** | **162** | **10 tables** |

This database is used to develop and test schema understanding, SQL generation, query execution, and assistant responses.

### Application Conversation Tables

The assistant also maintains its own conversation data.

**ai_conversations**

Stores conversation-level information:

- id
- title
- created_at
- updated_at

**ai_conversation_messages**

Stores individual messages:

- id
- conversation_id
- role
- content
- created_at

The messages table supports both user and assistant messages.

Successful assistant messages currently store the natural-language answer together with the executed SQL, allowing future requests to use previous query context.

The application conversation tables belong to the AI Assistant application, not the ERP business database.

---

## 6. Implemented Features

### 6.1 FastAPI Application Foundation

The application provides a modular FastAPI structure with versioned API routing.

Implemented:

- FastAPI application initialization.
- API router registration.
- Health-check endpoints.
- Swagger/OpenAPI documentation.
- Modular endpoint organization.

### 6.2 Ollama Integration

The application communicates with a locally running Ollama instance.

Implemented capabilities:

- Check Ollama connectivity.
- Retrieve available local models.
- Generate LLM responses.
- Select a model for generation.
- Configure the default model.

### 6.3 Database Integration

SQLAlchemy is used to connect the application to MySQL.

Implemented capabilities:

- Database engine initialization.
- Database connection management.
- SQLAlchemy session handling.
- Database schema inspection.
- Table and column discovery.
- Primary-key and foreign-key information retrieval.

### 6.4 Text-to-SQL Generation

The SQL Generation Service converts natural-language questions into MySQL SELECT queries.

The generation process uses:

- Database schema context.
- Table and column information.
- Relationships between tables.
- User question.
- LLM instructions.

The objective is to generate SQL based on the actual database structure instead of relying on invented table or column names.

### 6.5 SQL Validation and Security Foundation

SQLGlot is used to parse and validate generated SQL before execution.

Current validation foundation includes:

- SQL parsing using the MySQL dialect.
- Single-statement validation.
- SELECT/CTE query restrictions.
- Rejection of data modification statements.
- Rejection of unsupported INTO and locking operations.
- Database schema allowlisting.
- Table and column validation.
- Default query LIMIT.
- Maximum LIMIT restriction of 100 records.

The SQL Execution Service validates the query again before execution.

**Important:** These controls provide a development-stage security foundation. They do not yet represent a complete production security solution.

### 6.6 SQL Execution

The SQL Execution Service executes validated queries through SQLAlchemy.

It returns:

- Execution status.
- Execution message.
- Number of returned records.
- Query result data.

The service is shared between the standard SQL Assistant and Conversation Assistant.

### 6.7 Natural Language Answer Generation

After successful SQL execution, the Answer Generation Service sends the question, executed SQL, and query results to the LLM.

The LLM generates a readable answer based on the returned data.

Implemented capabilities:

- Explain database query results.
- Present multiple records as numbered lists.
- Include important fields when available.
- Avoid inventing information outside the returned results.
- Handle empty query results.

### 6.8 Main SQL Assistant

The standard SQL Assistant combines the following services:

```text
User Question
     |
     v
SQL Generation
     |
     v
SQL Validation
     |
     v
SQL Execution
     |
     v
Answer Generation
     |
     v
Final API Response
```

The main endpoint has been tested successfully through Swagger.

### 6.9 Conversation History and Context

A separate Conversation Assistant workflow has been implemented.

It supports:

- Creating a new conversation automatically.
- Continuing an existing conversation.
- Saving user messages.
- Saving assistant messages.
- Retrieving conversation history.
- Using previous messages as LLM context.
- Resolving references in follow-up questions.
- Saving executed SQL with assistant answers.
- Generating new SQL based on previous query context.

Example:

**First question**

```text
Show me the top 5 customers.
```

The assistant generates and executes a query that retrieves five customer records.

**Follow-up question**

```text
Show their email addresses only.
```

The assistant uses the previous conversation and executed SQL to understand which customers the user is referring to.

This avoids treating the follow-up as an unrelated database request.

The Conversation Assistant has been tested successfully through Swagger, including follow-up questions using the same conversation ID.

---

## 7. Available API Endpoints

The application exposes several endpoints organized by responsibility.

Base URL during local development:

```text
http://127.0.0.1:8000
```

Interactive API documentation:

```text
http://127.0.0.1:8000/docs
```

### 7.1 System Endpoints

| Method | Endpoint | Description |
|---|---|---|
| GET | `/` | Application welcome and status |
| GET | `/health` | Application health check |

**Use Case:**

Check whether the FastAPI application is running and responding.

Example:

```http
GET /health
```

Example response:

```json
{
    "status": "healthy",
    "application": "ERP AI Assistant"
}
```

### 7.2 Ollama Endpoints

| Method | Endpoint | Description |
|---|---|---|
| GET | `/api/v1/ollama/status` | Check Ollama connectivity |
| GET | `/api/v1/ollama/models` | Retrieve available local models |
| POST | `/api/v1/ollama/generate` | Generate a response using an Ollama model |

**Use Case:**

These endpoints help verify the LLM runtime and test model generation independently from the database assistant.

Example request:

```http
POST /api/v1/ollama/generate
```

```json
{
    "prompt": "Explain what artificial intelligence is.",
    "model": "qwen3:4b-instruct-2507-q4_K_M"
}
```

Example response structure:

```json
{
    "status": "success",
    "data": {
        "model": "qwen3:4b-instruct-2507-q4_K_M",
        "response": "Artificial intelligence is..."
    }
}
```

### 7.3 Database Schema Endpoint

| Method | Endpoint | Description |
|---|---|---|
| GET | `/api/v1/schema` | Retrieve database schema information |

**Use Case:**

Inspect the connected database structure and provide the LLM with the information needed to generate valid SQL.

The endpoint returns information about:

- Number of tables.
- Table names.
- Column names.
- Column data types.
- Primary keys.
- Foreign keys.

### 7.4 SQL Generation Endpoint

| Method | Endpoint | Description |
|---|---|---|
| POST | `/api/v1/sql/generate` | Generate SQL from a natural-language question |

**Use Case:**

Generate a SQL query independently without running the complete assistant workflow.

This is useful for testing and debugging Text-to-SQL generation.

The generated query must still pass through the SQL validation and execution controls before it is used against the database.

### 7.5 Main Assistant Endpoint

| Method | Endpoint | Description |
|---|---|---|
| POST | `/api/v1/assistant/ask` | Execute a complete SQL Assistant request |

**Use Case:**

This is the primary endpoint for users who want to ask database questions without managing the SQL generation and execution steps themselves.

Example request:

```json
{
    "question": "Show me the top 5 customers."
}
```

Example response structure:

```json
{
    "success": true,
    "question": "Show me the top 5 customers.",
    "generated_sql": "SELECT ...",
    "model": "qwen3:4b-instruct-2507-q4_K_M",
    "message": "Query executed successfully.",
    "row_count": 5,
    "data": [
        {
            "customer_id": 10,
            "customer_name": "Salma Tarek",
            "email": "salma@example.com",
            "phone": "01010000010",
            "city": "Cairo"
        }
    ],
    "answer": "Here are the top 5 customers..."
}
```

*The example is abbreviated; the actual response contains the returned records.*

### 7.6 Conversation Management Endpoints

| Method | Endpoint | Description |
|---|---|---|
| POST | `/api/v1/conversations/` | Create a new conversation |
| GET | `/api/v1/conversations/{conversation_id}` | Retrieve a conversation |
| GET | `/api/v1/conversations/{conversation_id}/messages` | Retrieve its messages |
| POST | `/api/v1/conversations/{conversation_id}/messages` | Add a message to a conversation |

**Use Case:**

These endpoints provide basic conversation management independently from the AI query workflow.

They can be used to create conversations, retrieve previous messages, and manage message records.

### 7.7 Conversation Assistant Endpoint

| Method | Endpoint | Description |
|---|---|---|
| POST | `/api/v1/conversation-assistant/ask` | Ask a question with conversation context |

**Use Case:**

This endpoint is designed for conversational database interaction.

It supports both new and existing conversations.

**Create a new conversation**

```json
{
    "question": "Show me the top 5 customers."
}
```

When `conversation_id` is omitted, a new conversation is created automatically.

**Continue an existing conversation**

```json
{
    "question": "Show their email addresses only.",
    "conversation_id": 2
}
```

The existing conversation history is retrieved and included in the SQL-generation context.

The response includes:

- success
- conversation_id
- question
- sql
- model
- message
- row_count
- data
- answer

### Endpoint Selection Guide

| Requirement | Endpoint |
|---|---|
| Check application status | `/health` |
| Check Ollama | `/api/v1/ollama/status` |
| List local models | `/api/v1/ollama/models` |
| Test LLM generation | `/api/v1/ollama/generate` |
| Inspect database structure | `/api/v1/schema` |
| Generate SQL only | `/api/v1/sql/generate` |
| Ask an independent database question | `/api/v1/assistant/ask` |
| Create or retrieve conversations | `/api/v1/conversations/` |
| Ask context-aware questions | `/api/v1/conversation-assistant/ask` |

---

## 8. Installation and Local Setup

### Prerequisites

Make sure the following are installed:

- Python 3.13.
- MySQL or MariaDB.
- Ollama.
- Git.

### Step 1: Clone the Repository

```bash
git clone https://github.com/mahmoud1641998/ERP_Assistant.git
```

Navigate to the project directory:

```bash
cd ERP_Assistant
```

### Step 2: Create a Virtual Environment

```bash
python -m venv .venv
```

Activate it on Windows:

```bash
.venv\Scripts\activate
```

### Step 3: Install Dependencies

```bash
pip install -r requirements.txt
```

### Step 4: Configure Environment Variables

Create a `.env` file in the project root.

Configure the database connection and Ollama settings according to the variable names defined in:

```text
app/core/config.py
```

Do not commit actual credentials or production database passwords to GitHub.

### Step 5: Start Ollama

Make sure the Ollama service is running.

Download the current development model:

```bash
ollama pull qwen3:4b-instruct-2507-q4_K_M
```

Verify the installed model:

```bash
ollama list
```

### Step 6: Start the FastAPI Application

```bash
uvicorn app.main:app --reload
```

### Step 7: Open Swagger Documentation

Navigate to:

```text
http://127.0.0.1:8000/docs
```

You can test all registered endpoints directly through Swagger UI.

---

## 9. Current Testing Status

The following functionality has been implemented and tested during development:

| Feature | Status |
|---|---|
| FastAPI application startup | Completed |
| Ollama connectivity | Tested |
| Local model retrieval | Tested |
| Ollama response generation | Tested |
| MySQL connection | Tested |
| Database schema inspection | Tested |
| Text-to-SQL generation | Tested |
| SQL validation foundation | Tested |
| SQL query execution | Tested |
| Natural-language answer generation | Tested |
| Main Assistant API | End-to-end tested |
| Conversation creation and retrieval | Tested |
| Conversation message storage | Tested |
| Context-aware SQL generation | Tested |
| Conversation Assistant API | End-to-end tested |
| Follow-up questions | Tested |
| Previous SQL storage in conversation history | Tested |

Testing so far has primarily used local development services and Swagger UI.

A comprehensive automated test suite and production security assessment are still planned.

---

## 10. Current Limitations and Security Considerations

The current implementation is a development-stage foundation, not a production-ready deployment.

Before connecting it to a live ERP database, the following areas must be addressed.

### Database Security

- Use a dedicated database account with read-only permissions for ERP business data.
- Never give the AI Assistant unrestricted write access.
- Apply database-level permissions in addition to application-level SQL validation.
- Restrict accessible tables and columns according to business requirements.

### SQL Security

- Strengthen SQL validation against potentially dangerous functions and query patterns.
- Add query execution timeouts.
- Improve error handling and logging.
- Review generated SQL for expensive operations.
- Add stricter access control for sensitive information.

### Conversation Security

- Add user authentication.
- Associate conversations with their owners.
- Prevent users from accessing other users' conversations.
- Introduce tenant isolation when multiple ERP clients are supported.
- Review conversation message management permissions.

### Production Infrastructure

- Configure production environment variables securely.
- Avoid exposing Ollama directly to public clients.
- Add API authentication and authorization.
- Configure network access between the Laravel application and FastAPI.
- Monitor model response time and resource consumption.

---

## 11. Future Development Roadmap

The next development stages are planned as follows.

### Phase 1: Conversation Assistant Stabilization

- Review and improve API error handling.
- Standardize error responses.
- Improve conversation management.
- Add conversation listing.
- Add conversation deletion and lifecycle management.
- Review message ordering and pagination.
- Improve conversation metadata and titles.
- Add automated tests using Pytest.

### Phase 2: Security and Production Readiness

- Read-only database credentials.
- User authentication and authorization.
- Tenant isolation.
- Stronger SQL validation.
- Query execution timeouts.
- Rate limiting.
- Better logging and monitoring.
- Sensitive-data access restrictions.

### Phase 3: Document Processing

Allow users to upload business documents such as:

- PDF files.
- Business policies.
- Product documentation.
- Internal manuals.
- Operational procedures.

Implement document extraction, text cleaning, chunking, and metadata handling.

### Phase 4: Embeddings and Vector Search

Introduce:

- Embedding models.
- Vector storage.
- Semantic similarity search.
- Document retrieval.
- Metadata filtering.

### Phase 5: RAG Integration

Build a Retrieval-Augmented Generation pipeline to allow the assistant to answer questions using internal ERP documentation.

Example:

```text
User Question
      |
      v
Embedding Generation
      |
      v
Vector Search
      |
      v
Relevant Document Chunks
      |
      v
LLM Context Construction
      |
      v
Natural Language Answer
```

The future system will support both database-based answers and document-based answers.

### Phase 6: Advanced ERP AI Capabilities

Potential future features:

- Sales summaries.
- Customer analytics.
- Inventory insights.
- Supplier analysis.
- Financial reporting.
- Business intelligence questions.
- Automated report generation.
- Role-aware AI assistance.

### Phase 7: Deployment and Integration

- Production deployment.
- API authentication.
- Docker-based packaging if needed.
- Environment-specific configuration.
- Monitoring and logging.
- Laravel ERP integration.
- Performance optimization.

---

## 12. Integration with the Existing Laravel ERP System

One of the main objectives is to integrate the independent FastAPI AI Assistant with an existing Laravel ERP application.

The ERP system already has its own backend and MySQL database.

The AI Assistant should work as a separate service and communicate with Laravel through HTTP APIs.

### 12.1 Target Integration Architecture

```text
                ERP User
                    |
                    v
             Laravel Frontend
                    |
                    v
              Laravel Backend
                    |
                    | HTTP Request
                    v
          FastAPI AI Assistant
                    |
                    v
              Ollama / Qwen3
                    |
                    v
              SQL Generation
                    |
                    v
              SQL Validation
                    |
                    v
           ERP MySQL Database
             (Read-Only User)
                    |
                    v
              Query Results
                    |
                    v
             Answer Generation
                    |
                    v
              FastAPI Response
                    |
                    v
             Laravel Backend
                    |
                    v
              ERP Frontend
```

### 12.2 Integration Principle

The AI Assistant remains independent from Laravel.

Laravel does not need to implement the LLM, SQL generation, or conversation logic.

Instead:

1. Laravel receives a question from the ERP user.
2. Laravel sends the question to the FastAPI service.
3. FastAPI processes the question.
4. FastAPI accesses the authorized ERP database connection.
5. FastAPI generates and executes the validated SQL.
6. FastAPI returns the results and natural-language answer.
7. Laravel displays the response to the user.

### 12.3 Step 1: Connect the AI Assistant to the ERP Database

The current development database is:

```text
erp_ai_assistant_test
```

For integration, a separate database connection must be configured for the actual ERP database.

The FastAPI application should have its own database connection settings.

The production connection must use a dedicated MySQL account with SELECT-only permissions on the approved ERP tables.

**Important:** The AI Assistant must not use Laravel's database credentials or inherit unrestricted ERP database privileges.

The AI service should connect to the ERP database through its own controlled credentials.

### 12.4 Step 2: Configure the ERP Database Schema

The AI Assistant needs access to the ERP database structure.

The schema service should inspect the approved tables and retrieve:

- Table names.
- Column names.
- Data types.
- Primary keys.
- Foreign keys.
- Relationships.

The schema context should be limited to the tables the AI Assistant is allowed to access.

This information is essential for generating valid SQL queries.

### 12.5 Step 3: Configure Laravel to Communicate with FastAPI

Add the FastAPI service URL to Laravel's environment configuration.

Example:

```env
AI_ASSISTANT_URL=http://127.0.0.1:8000
AI_ASSISTANT_API_KEY=your-secure-api-key
```

The actual URL must point to the deployed FastAPI service when the applications run on separate servers.

The API key is a planned integration security mechanism and must be implemented and validated by the FastAPI application before relying on it.

### 12.6 Step 4: Create a Laravel Service

Create a dedicated service responsible for communicating with the FastAPI application.

Example file:

```text
app/Services/AiAssistantService.php
```

Example implementation:

```php
<?php

namespace App\Services;

use Illuminate\Support\Facades\Http;

class AiAssistantService
{
    public function ask(
        string $question,
        ?int $conversationId = null
    ): array {

        $payload = [
            'question' => $question,
        ];

        if ($conversationId !== null) {
            $payload['conversation_id'] = $conversationId;
        }

        $response = Http::timeout(120)
            ->withHeaders([
                'X-API-Key' => config('services.ai_assistant.api_key'),
            ])
            ->post(
                rtrim(config('services.ai_assistant.url'), '/')
                . '/api/v1/conversation-assistant/ask',
                $payload
            );

        $response->throw();

        return $response->json();
    }
}
```

This service sends the user's question to the Conversation Assistant endpoint and returns the JSON response.

The timeout shown is an initial example and should be adjusted after measuring real model response times.

### 12.7 Step 5: Configure Laravel Services

In:

```text
config/services.php
```

Add:

```php
'ai_assistant' => [
    'url' => env('AI_ASSISTANT_URL'),
    'api_key' => env('AI_ASSISTANT_API_KEY'),
],
```

This allows the Laravel application to retrieve the AI Assistant configuration securely through Laravel's configuration system.

### 12.8 Step 6: Create a Laravel Controller

Create a controller responsible for receiving the user's question and calling the AI Assistant service.

Example file:

```text
app/Http/Controllers/AiAssistantController.php
```

Example implementation:

```php
<?php

namespace App\Http\Controllers;

use Illuminate\Http\Request;
use App\Services\AiAssistantService;

class AiAssistantController extends Controller
{
    public function ask(
        Request $request,
        AiAssistantService $assistant
    ) {

        $validated = $request->validate([
            'question' => 'required|string|min:3|max:1000',
            'conversation_id' => 'nullable|integer',
        ]);

        $result = $assistant->ask(
            $validated['question'],
            $validated['conversation_id'] ?? null
        );

        return response()->json($result);
    }
}
```

### 12.9 Step 7: Add a Laravel API Route

In:

```text
routes/api.php
```

Add:

```php
use App\Http\Controllers\AiAssistantController;

Route::middleware('auth:sanctum')->post(
    '/ai-assistant/ask',
    [AiAssistantController::class, 'ask']
);
```

The authentication middleware should match the ERP application's actual authentication system.

### 12.10 Step 8: Connect the ERP Frontend

The frontend can provide a chat interface where users:

- Enter natural-language questions.
- Receive assistant responses.
- View query results.
- Continue previous conversations.
- Maintain conversation IDs between requests.

Example request from the ERP frontend:

```json
{
    "question": "Show me the top 5 customers."
}
```

Example response handling:

```json
{
    "success": true,
    "conversation_id": 15,
    "question": "Show me the top 5 customers.",
    "sql": "SELECT ...",
    "model": "qwen3:4b-instruct-2507-q4_K_M",
    "message": "Query executed successfully.",
    "row_count": 5,
    "data": [],
    "answer": "Here are the top 5 customers..."
}
```

Laravel should preserve the returned `conversation_id` and send it with subsequent questions in the same conversation.

### 12.11 Step 9: Implement Access Control

Before production integration, the system must enforce:

- Authenticated ERP users only.
- Conversation ownership.
- Role-based table access.
- Tenant-specific database access where applicable.
- Restricted access to sensitive customer and financial information.
- API-to-API authentication.
- Database-level read-only permissions.

The FastAPI application must not trust the user's natural-language request as an authorization mechanism.

Access control must be enforced independently of the LLM.

### 12.12 Step 10: Integration Testing

Test the complete workflow:

1. Open the Laravel ERP application.
2. Authenticate as an authorized ERP user.
3. Open the AI Assistant interface.
4. Ask a question about ERP data.
5. Verify that Laravel sends the request to FastAPI.
6. Verify that FastAPI generates and validates SQL.
7. Verify that the query reads from the intended ERP database.
8. Verify that the answer is returned to Laravel.
9. Ask a follow-up question.
10. Verify that the same conversation context is preserved.
11. Test unauthorized access and invalid requests.
12. Confirm that no database modification can be performed through the assistant.

---

## 13. Final Project Vision

The long-term objective is to transform the ERP system into an AI-assisted business platform.

Instead of requiring users to understand database structures, navigate multiple modules, or request reports manually, they will be able to interact with ERP data through natural language.

The final system is intended to combine:

- **Text-to-SQL:** Natural-language database interaction.
- **Conversational AI:** Context-aware follow-up questions.
- **RAG:** Retrieval from internal business documents.
- **Business Intelligence:** Natural-language reporting and insights.
- **ERP Integration:** Direct interaction with existing business workflows through controlled APIs.

The development strategy is incremental: build, test, validate, secure, and then integrate each capability before moving to the next stage.

---

## Project Repository

GitHub:

[ERP_Assistant](https://github.com/mahmoud1641998/ERP_Assistant)

**Current Development Stage:** Conversation History & Context — stabilization and API review.

**Next Development Focus:** Error handling, conversation management, automated testing, and production security.